# Notebook 51 — Capstone III: Operate a Durable Multi-Agent System

    ## Learning objectives

    - Combine durable state, MCP tools, approvals, and specialist agents
- Evaluate trajectories, failure recovery, security, and capacity
- Produce an operational release with rollback and governance

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 51.1 Workflow architecture

Choose a consequential but sandboxable task and first implement a deterministic workflow. Add one agent only for ambiguous decisions, then justify any specialist agents through context or permission isolation. Define typed state, events, task envelopes, tool schemas, terminal states, global budgets, and owners. Use MCP for composable capabilities without treating discovered servers as trusted. Draw every identity, data, and side-effect boundary before executing.


In [ ]:
architecture={"host":"policy + durable state","servers":["catalog","sandbox"],"agents":["supervisor","researcher","reviewer"],"human_gate":"external writes"}; print(architecture)


## 51.2 Durability and multi-agent coordination

Checkpoint before and after effects, issue stable idempotency keys, retain receipts, propagate cancellation, and resume after injected crashes. Human approval binds to an immutable proposed action and expires. A supervisor may route only registered roles; specialists receive least context and tools; parallel results merge deterministically with dissent preserved. Shared state is authoritative only through validated transitions, not agent-authored prose.


In [ ]:
states=["ready","planning","running","waiting_approval","committing","done","failed","cancelled"]; print(states)


## 51.3 Evaluation and adversarial testing

Create a deterministic environment and frozen tasks. Compare deterministic, single-agent, and multi-agent variants under equal model, permission, token, tool-call, and time budgets. Score task success, tool choice and arguments, effects, duplicated work, conflicts, intervention, steps, tokens, latency, and critical violations. Inject malformed calls, poisoned MCP resources, revoked permissions, duplicate delivery, worker loss, stale approval, sandbox attacks, and malicious specialists.


In [ ]:
faults=["crash_before_effect","crash_after_effect","malicious_resource","revoked_scope","worker_timeout","duplicate_message"]; print(faults)


## 51.4 Deployment and governance

Serve the pinned model through vLLM or the selected engine behind authenticated ingress and admission control. Trace state transitions and effects without retaining sensitive payloads unnecessarily. Canary with read-only scopes, rehearse rollback, and publish a system card containing versions, architecture, threat model, evaluations, known limitations, monitoring, ownership, and incident response. Conclude with an ablation-based answer to whether multiple agents improved the workflow enough to remain.


In [ ]:
comparison={"deterministic":.62,"single_agent":.81,"multi_agent":.84,"multi_agent_cost_ratio":2.3}; print(comparison,"keep only if bounded gains justify cost")


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## Exercises

    1. Implement crash and resume tests.
2. Run matched-budget architecture ablations.
3. Publish a system card and incident drill.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
